# Adaptive Trust Gate — Goodreads Poetry, unattended overnight run

Press **Runtime → Run all**, answer the two prompts in the first three cells, then
leave it. Nothing after cell 4 asks for anything.

What it does, in order: full single-seed pipeline (7 models + 4 external baselines +
comparison + interpretability), then the multi-seed run for error bars.

**Results save themselves.** A background thread copies `results/` to your Drive every
3 minutes, into a fresh timestamped folder — your existing `atg_results/goodreads_poetry`
is never touched. If the runtime disconnects overnight you lose at most 3 minutes of
output, and the multi-seed script checkpoints after every seed, so re-running resumes
from where it stopped.

Budget roughly 1 hour for the single-seed pipeline and 40–60 minutes per seed after
that.


## 1. Code and dependencies


In [ ]:
%cd /content
!git clone -q https://github.com/Ar555Rathod/adaptive-trust-gate.git 2>/dev/null || git -C adaptive-trust-gate pull -q origin master
%cd /content/adaptive-trust-gate
!git log --oneline -1


In [ ]:
!pip install -q -r requirements.txt
try:
    import surprise
    print('scikit-surprise OK:', surprise.__version__)
except Exception as e:
    print('scikit-surprise FAILED:', e)
    print('STOP -- fix this before leaving the run unattended.')


## 2. Mount Drive — the only prompt in this notebook

Approve the Google sign-in popup when it appears. Everything after this runs
without asking for anything.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 3. Start the auto-save thread

Copies `results/` to Drive every 3 minutes in the background. It is **additive** —
no `--delete` — and writes into a new timestamped folder, so nothing already in
your Drive can be overwritten or removed by this run.


In [ ]:
import shutil, subprocess, threading, time
from pathlib import Path

SRC = Path('/content/adaptive-trust-gate/results')
RUN_ID = time.strftime('run_%Y%m%d_%H%M')
DST = Path('/content/drive/MyDrive/atg_results') / RUN_ID
DST.mkdir(parents=True, exist_ok=True)

HAVE_RSYNC = shutil.which('rsync') is not None
sync_state = {'count': 0, 'last': None, 'error': None}

def sync_once():
    if not SRC.exists() or not any(SRC.rglob('*')):
        return False
    if HAVE_RSYNC:
        subprocess.run(['rsync', '-a', f'{SRC}/', f'{DST}/'], check=False, capture_output=True)
    else:
        shutil.copytree(SRC, DST, dirs_exist_ok=True)
    return True

def sync_loop(every=180):
    while True:
        try:
            if sync_once():
                sync_state['count'] += 1
                sync_state['last'] = time.strftime('%H:%M:%S')
        except Exception as e:
            sync_state['error'] = repr(e)
        time.sleep(every)

threading.Thread(target=sync_loop, daemon=True).start()
print(f'auto-save ON  ->  {DST}')
print(f'   method: {"rsync" if HAVE_RSYNC else "copytree"}   interval: 180s')
print('   additive only -- existing Drive folders are not modified')


## 4. Dataset


In [ ]:
BASE = 'https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre'
!mkdir -p data/raw/goodreads
!wget -q -O data/raw/goodreads/goodreads_books_poetry.json.gz {BASE}/goodreads_books_poetry.json.gz
!wget -q -O data/raw/goodreads/goodreads_interactions_poetry.json.gz {BASE}/goodreads_interactions_poetry.json.gz
!ls -lh data/raw/goodreads


In [ ]:
%env ATG_DATASET=goodreads
%env ATG_GOODREADS_GENRE=poetry
%env PYTHONPATH=src


## 5. Single-seed pipeline (seed 42)


In [ ]:
!python src/atg/data/normalize.py
!python scripts/01_build_splits.py


In [ ]:
!python scripts/02_train_experts.py


In [ ]:
!python scripts/03_static_hybrid.py
!python scripts/04_learned_gate.py
!python scripts/05_bandit_gate.py


In [ ]:
!python scripts/06_ga_gate.py
!python scripts/07_sequential_gate.py


In [ ]:
!python scripts/11_external_baselines.py
!python scripts/08_full_comparison.py
!python scripts/09_interpretability.py


## 6. Multi-seed run — the long one

All 11 models (7 gates/experts + 4 baselines) re-fitted per seed, so every row in
the final table gets mean ± std.

Checkpoints after each seed and resumes on restart, so a disconnect costs one seed,
not the run. The retry loop below restarts it automatically if a seed crashes.


In [ ]:
# Resume across runtimes. The multi-seed script skips seeds already recorded in
# its checkpoint file, but a fresh Colab runtime starts with an empty results/,
# so without this a disconnect overnight means re-running every seed. Copies the
# checkpoint back from the most recent previous run folder in Drive, if any.
from pathlib import Path
import shutil

root = Path('/content/drive/MyDrive/atg_results')
prev = sorted(
    (q for q in root.glob('run_*/goodreads_poetry/metrics/multiseed_full_comparison.json')
     if q.parent.parent.parent.name != RUN_ID),
    key=lambda q: q.stat().st_mtime, reverse=True)

if prev:
    import json as _json
    done = _json.load(open(prev[0])).get('seeds', [])
    dest = Path('/content/adaptive-trust-gate/results/goodreads_poetry/metrics')
    dest.mkdir(parents=True, exist_ok=True)
    shutil.copy2(prev[0], dest / 'multiseed_full_comparison.json')
    print(f'restored checkpoint from {prev[0].parent.parent.parent.name}: seeds {done} already done')
else:
    print('no previous checkpoint in Drive -- starting the seed run from scratch')


In [ ]:
import subprocess, os, time

env = dict(os.environ, ATG_SEEDS='42,1,2')

for attempt in range(1, 4):
    print(f'--- multiseed attempt {attempt}  ({time.strftime("%H:%M:%S")}) ---', flush=True)
    r = subprocess.run(['python', '-u', 'scripts/10_multiseed_full.py'], env=env)
    if r.returncode == 0:
        print('multiseed finished cleanly')
        break
    print(f'exit code {r.returncode} -- retrying; completed seeds are checkpointed')
else:
    print('multiseed did not finish after 3 attempts -- partial results are still saved')


## 7. Final save and verification


In [ ]:
ok = sync_once()
print(f'final sync: {"done" if ok else "nothing to copy"}')
print(f'syncs during run: {sync_state["count"]}   last: {sync_state["last"]}   error: {sync_state["error"]}')
print(f'\nsaved to: {DST}\n')
!ls -R {DST} | head -50


In [ ]:
# Morning check: did the multi-seed run actually produce error bars?
import json
p = f'{DST}/goodreads_poetry/metrics/multiseed_full_comparison.json'
try:
    d = json.load(open(p))
    print('seeds completed:', d['seeds'], '\n')
    print(f"{'model':22s} {'overall RMSE':>18s}")
    for name, agg in d['aggregate'].items():
        a = agg['overall']
        print(f"{name:22s} {a['rmse_mean']:.4f} +/- {a['rmse_std']:.4f}")
except FileNotFoundError:
    print('not found -- the multi-seed cell did not complete:', p)
